# PA algos

In [ ]:
import os
import pandas as pd

def check_results_with_eoy(base_dir):
    combined_info = []

    # 1. base_dir 내의 모든 항목(폴더) 확인
    for folder_name in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, folder_name)

        if os.path.isdir(folder_path):
            # 파일 경로들
            csv_path = os.path.join(folder_path, "PA_ALL_TRADES.csv")
            html_path = os.path.join(folder_path, "PA_report.html")
            
            # A. CSV 파일 크기 확인
            size_mb = None
            if os.path.exists(csv_path):
                size_mb = os.path.getsize(csv_path) / (1024 * 1024)
            
            # B. HTML에서 EOY Returns 테이블 추출
            eoy_data = "Not Found"
            if os.path.exists(html_path):
                try:
                    # HTML 내의 모든 테이블 읽기
                    tables = pd.read_html(html_path)
                    
                    # QuantStats 리포트에서 보통 'Year', 'Return' 컬럼이 있는 테이블이 EOY 테이블임
                    for df in tables:
                        if 'Year' in df.columns and 'Return' in df.columns:
                            # 보기 좋게 문자열로 변환 (예: 2023: 371%, 2024: 121%)
                            eoy_list = []
                            for _, row in df.iterrows():
                                eoy_list.append(f"{int(row['Year'])}: {row['Return']}")
                            eoy_data = " | ".join(eoy_list)
                            break
                except Exception as e:
                    eoy_data = f"Error: {str(e)}"

            combined_info.append({
                "folder": folder_name,
                "size_mb": size_mb,
                "eoy": eoy_data
            })

    return combined_info

# --- 실행 부분 ---
ROOT_PATH = "/Users/cyberedjs/Desktop/Unity/results/pa_trade_results/Momentum/hold" 
results = check_results_with_eoy(ROOT_PATH)

# 정렬: 사이즈 큰 순서대로
results.sort(key=lambda x: x['size_mb'] if x['size_mb'] is not None else 0, reverse=True)

# --- 결과 출력 부분 수정 ---

# 1. 헤더 설정 (너비 지정: 폴더명 40자, 사이즈 15자, EOY 40자)
header_format = "{:<40} | {:>12} | {:<40}"
separator = "-" * 100

print("\n" + separator)
print(header_format.format("Folder Name", "Size (MB)", "EOY Returns (Performance)"))
print(separator)

# 2. 데이터 출력
for item in results:
    # 사이즈 처리 (None 대응 및 포맷팅)
    if item['size_mb'] is not None:
        size_str = f"{item['size_mb']:>9.2f} MB"
    else:
        size_str = "Not Found"
    
    # 폴더명이나 EOY가 너무 길 경우를 대비해 슬라이싱 처리 (선택 사항)
    folder_name = (item['folder'][:37] + '..') if len(item['folder']) > 37 else item['folder']
    eoy_str = item['eoy']

    # 설정한 포맷에 맞춰 출력
    print(header_format.format(folder_name, size_str, eoy_str))

print(separator)

## Selection algos

In [ ]:
import os
import pandas as pd

# 1. 설정: 결과가 저장된 기본 경로 (사용자의 환경에 맞게 수정하세요)
# 예: ./results/pre_selection_eval/hold 또는 ./results/hold
STRATEGY_NAME = "Volume-Validated Impulse EMA Momentum (VVIEM)"
RESULT_DIR = f"/Users/cyberedjs/Desktop/Unity/results/backtest_results/{STRATEGY_NAME}"
BASE_RESULT_DIR = os.path.join(RESULT_DIR, "hold")

def show_aggregated_metrics(base_dir):
    print(f"\n📂 Loading metrics from: {base_dir}")
    
    if not os.path.exists(base_dir):
        print(f"❌ 경로가 존재하지 않습니다: {base_dir}")
        return

    all_metrics = []

    # 2. 하위 폴더 순회하며 Summary_Metrics.csv 찾기
    for algo_folder in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, algo_folder)
        
        # 폴더인 경우에만 진입
        if os.path.isdir(folder_path):
            summary_file = os.path.join(folder_path, "Summary_Metrics.csv")
            
            if os.path.exists(summary_file):
                try:
                    df = pd.read_csv(summary_file)
                    all_metrics.append(df)
                except Exception as e:
                    print(f"⚠️ 읽기 실패 ({algo_folder}): {e}")

    # 3. 데이터가 없으면 종료
    if not all_metrics:
        print("❌ 저장된 Summary Metrics 파일이 없습니다.")
        return

    # 4. 하나의 데이터프레임으로 병합
    final_df = pd.concat(all_metrics, ignore_index=True)

    # 5. 정렬 (Ranking Logic)
    # 1순위: Hit Rate (높을수록 좋음)
    # 2순위: MAR (낮을수록 좋음) -> Hit Rate가 비슷하면 순위가 높은 게 승리
    final_df = final_df.sort_values(
        by=["Total_Trades", "Avg_MAR", "Avg_Hit_Rate"], 
        ascending=[True, False, False] 
    ).reset_index(drop=True)

    # 6. 보기 좋게 출력 옵션 설정
    pd.set_option('display.max_rows', None)      # ★ 핵심: 행(Row)이 아무리 많아도 생략하지 않음
    pd.set_option('display.max_columns', None)  # 모든 컬럼 보기
    pd.set_option('display.width', 1000)        # 줄바꿈 방지
    pd.set_option('display.max_colwidth', None)  # ★ 추가: 내용(Logic 등)이 길어도 '...'으로 자르지 않음
    pd.set_option('display.colheader_justify', 'center') # 헤더 가운데 정렬
    pd.set_option('display.float_format', '{:.4f}'.format) # 소수점 포맷

    print("\n" + "="*80)
    print("🏆 Selection Algorithm Leaderboard")
    print("="*80)
    
    # Logic 컬럼은 너무 길 수 있으니 화면 출력에선 제외하거나 잘라서 보여주기
    display_df = final_df.copy()
    if 'Logic' in display_df.columns:
        # Logic이 너무 길면 50자로 자름
        display_df['Logic'] = display_df['Logic'].apply(lambda x: (x[:47] + '...') if isinstance(x, str) and len(x) > 50 else x)

    print(display_df)
    print("="*80 + "\n")

    return final_df

if __name__ == "__main__":
    # 실행
    df_result = show_aggregated_metrics(BASE_RESULT_DIR)
    
    # # 필요하다면 결과를 엑셀이나 CSV로 저장
    # if df_result is not None:
    #     df_result.to_csv("Final_Leaderboard.csv", index=False)
    #     print("✅ 전체 요약 결과가 'Final_Leaderboard.csv'로 저장되었습니다.")

## Fast Backtest

In [2]:
!pwd

/Users/cyberedjs/Desktop/Unity/src


In [ ]:
import os
import ast
import numpy as np
import pandas as pd
import quantstats as qs
import matplotlib.pyplot as plt

# ==========================================
# 📂 파일 경로 설정 (사용자 환경에 맞게 수정하세요)
# ==========================================
PA_FILE = "/Users/cyberedjs/Desktop/Unity/results/pa_trade_results/Momentum/priority/Volume-Filtered Channel Momentum with Dual SMA Trend Filter/PA_All_Trades.csv"
STRATEGY_NAME = "Volume-Validated Impulse EMA Momentum (VVIEM)"
INDICATOR_NAME = "EMA-Buffered_Momentum_Quality_Score"
REPORT_FILE = f"/Users/cyberedjs/Desktop/Unity/results/backtest_results/{STRATEGY_NAME}/hold/{INDICATOR_NAME}"
SELECTION_FILE = f"/Users/cyberedjs/Desktop/Unity/results/backtest_results/{STRATEGY_NAME}/hold/{INDICATOR_NAME}/Selection_Evaluation_Trades.csv"
# ==========================================

def run_fast_backtest():
    print("🚀 Starting Fast Backtest...")

    # 1. 파일 존재 확인 및 로드
    if not os.path.exists(SELECTION_FILE) or not os.path.exists(PA_FILE):
        print(f"❌ 파일을 찾을 수 없습니다.\nSelection: {SELECTION_FILE}\nTrades: {PA_FILE}")
        return

    print(f"📥 Loading Selection Log...")
    df_select = pd.read_csv(SELECTION_FILE)
    
    print(f"📥 Loading Trade Log...")
    df_trades = pd.read_csv(PA_FILE)

    # 2. 데이터 전처리 (날짜 형식 통일)
    # 이미지 상 포맷이 '2023.6.1 12:00' 형식이므로 pandas가 자동으로 추론하게 함
    print("⚙️  Preprocessing Timestamps...")
    
    df_select['timestamp'] = pd.to_datetime(df_select['timestamp'])
    
    # PA Trade 파일의 'anchor' 컬럼 이용 (entry_time 가공 불필요)
    df_trades['anchor'] = pd.to_datetime(df_trades['anchor'])
    df_trades['exit_time'] = pd.to_datetime(df_trades['exit_time'])

    # 3. 문자열로 된 리스트 변환 ("['BTC', 'ETH']" -> ['BTC', 'ETH'])
    print("⚙️  Parsing Selected Symbols...")
    df_select['selected_symbols'] = df_select['selected_symbols'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else []
    )

    # 4. 데이터 병합 (Inner Join)
    # PA Trade 데이터에 Selection 정보를 'Anchor(시간)' 기준으로 붙임
    print("🔗 Merging Data based on Anchor Time...")
    merged_df = pd.merge(
        df_trades, 
        df_select[['timestamp', 'selected_symbols']], 
        left_on='anchor', 
        right_on='timestamp', 
        how='inner'
    )

    # 5. 핵심 필터링 로직
    # "이 거래의 종목(symbol)이, 그 시간대의 선정 종목 리스트(selected_symbols)에 들어있는가?"
    # *참고: 이 방식은 한 앵커에 같은 종목 거래가 3건이어도 3건 모두 살려냅니다.
    print("🔍 Filtering Trades (Matching Symbol with Selection)...")

    for n in range(6, 11): # 1, 2, 3, 4, 5
        print(f"\n🔍 Testing TOP {n} Selection...")

        # Row별 필터링 함수
        def is_in_top_n(row):
            # 저장된 리스트에서 앞의 n개만 자름
            target_list = row['selected_symbols'][:n] 
            return row['symbol'] in target_list

        final_df = merged_df[merged_df.apply(is_in_top_n, axis=1)].copy()

        print(f"✅ Filtered Trades: {len(df_trades)} -> {len(final_df)} (Selection Logic Applied)")

        if len(final_df) == 0:
            print("❌ 선택된 알고리즘에 의해 체결된 거래가 하나도 없습니다.")
            return

        # 6. 수익 곡선 계산 (요청하신 로직 반영)
        # 거래 종료 시간 순 정렬
        final_df['exit_time'] = pd.to_datetime(final_df['exit_time'])
        aum=(final_df.set_index('exit_time').sort_index()['return']*1000).cumsum()+10000
        aum_daily = aum.resample('D').last().ffill()

        html_path = os.path.join(REPORT_FILE, f"Selection_Report_TOP{n}.html")
        try:
            qs.reports.html(aum.pct_change(), output=html_path, title=f"Selection Strategy TOP{n} Report")
            print(f"✅ 최종 리포트 생성 완료: {html_path}")
        except Exception as e:
            print(f"[Error] 리포트 생성 실패: {e}")

        final_df.to_csv(f"/Users/cyberedjs/Desktop/Unity/results/backtest_results/{STRATEGY_NAME}/hold/{INDICATOR_NAME}/Selection_Trades_TOP{n}.csv")

    #return final_df, aum_daily
    
if __name__ == "__main__":
    run_fast_backtest()
    # result, aum = run_fast_backtest()

✅ 최종 리포트 생성 완료: /Users/cyberedjs/Desktop/Unity/results/backtest_results/Volume-Validated Impulse EMA Momentum (VVIEM)/hold/EMA-Buffered_Momentum_Quality_Score/Selection_Report_9.html
